# SignEase: ASL Citizen → hand landmarks

Turns the ASL Citizen videos into the same MediaPipe Holistic landmarks the
SignEase app uses, and saves **only the landmarks** (small files) to Google
Drive. The 46 GB zip stays on this Colab machine and is never copied to Drive.

Run the cells top to bottom (Runtime → Run all works after step 1).

* **Step 1** installs the exact MediaPipe version and restarts the session (the
  "session crashed" message is expected). Then run the remaining cells.
* If Colab disconnects, just run everything again: finished chunks in Drive are
  skipped, so it continues where it stopped.

In [ ]:
# Step 1: install the same MediaPipe the SignEase training used, then restart.
# Takes 1-3 minutes. Only ready-made packages are used, so it never sits
# for ages compiling; if the pinned versions do not exist for this Python it
# falls back to the newest MediaPipe.
import sys, subprocess, os
print("Python", sys.version.split()[0], flush=True)
def pip(*packages):
    return subprocess.call([sys.executable, "-m", "pip", "install", "--only-binary=:all:",
                            "--progress-bar", "off", *packages])
if pip("numpy<2", "mediapipe==0.10.14", "opencv-python-headless==4.10.0.84") != 0:
    print("Pinned versions are not available for this Python; installing the newest mediapipe.", flush=True)
    if pip("mediapipe", "opencv-python-headless") != 0:
        raise SystemExit("No MediaPipe package for this Python. Runtime > Change runtime type > pick an older runtime version, then run again.")
print("Installed. Restarting the session now ('session crashed' is expected); then run step 2 and below.", flush=True)
os.kill(os.getpid(), 9)

In [ ]:
# Step 2: settings. Paste the ASL Citizen download link you got after
# accepting the license (right-click the download button → Copy link).
ZIP_URL = "PASTE_THE_DOWNLOAD_LINK_HERE"

# "ours": only videos of the 250 signs SignEase already knows (do this first).
# "all":  then every other sign too, to grow the vocabulary (much longer; run
#         it in later sessions, it resumes).
PHASE = "ours"

DRIVE_DIR = "/content/drive/MyDrive/SignEase/asl_citizen"
ZIP_PATH = "/content/ASL_Citizen.zip"   # this machine's disk, not Drive
CHUNK = 400                              # videos per saved file
MAX_WIDTH = 640                          # same as the SignEase backend

from google.colab import drive
drive.mount("/content/drive")
import os, shutil
os.makedirs(DRIVE_DIR, exist_ok=True)
import mediapipe as mp, cv2, numpy as np
assert hasattr(mp, "solutions"), "This MediaPipe version has no Holistic; pick an older runtime version (Runtime > Change runtime type) and rerun step 1."
print("mediapipe", mp.__version__, "| opencv", cv2.__version__, "| numpy", np.__version__)
print("free disk on this machine: %.0f GB" % (shutil.disk_usage("/content").free / 1e9))

In [ ]:
# Step 3: download the zip to this machine (about 46 GB; resumes if interrupted).
import os, subprocess, zipfile
assert ZIP_URL.startswith("http"), "Paste the download link into ZIP_URL in step 2."
def zip_ok(path):
    try:
        with zipfile.ZipFile(path) as z:
            return len(z.namelist()) > 0
    except Exception:
        return False
if not zip_ok(ZIP_PATH):
    subprocess.check_call(["curl", "-L", "--fail", "--retry", "5", "-C", "-", "-o", ZIP_PATH, ZIP_URL])
assert zip_ok(ZIP_PATH), "The download is not a valid zip. Check the link (it may have expired) and run again."
print("zip ready: %.1f GB" % (os.path.getsize(ZIP_PATH) / 1e9))

In [ ]:
# Step 4: read the dataset index and decide the order (SignEase's signs first).
import io, re, zipfile, pandas as pd
SIGNEASE_SIGNS = ["TV", "after", "airplane", "all", "alligator", "animal", "another", "any", "apple", "arm", "aunt", "awake", "backyard", "bad", "balloon", "bath", "because", "bed", "bedroom", "bee", "before", "beside", "better", "bird", "black", "blow", "blue", "boat", "book", "boy", "brother", "brown", "bug", "bye", "callonphone", "can", "car", "carrot", "cat", "cereal", "chair", "cheek", "child", "chin", "chocolate", "clean", "close", "closet", "cloud", "clown", "cow", "cowboy", "cry", "cut", "cute", "dad", "dance", "dirty", "dog", "doll", "donkey", "down", "drawer", "drink", "drop", "dry", "dryer", "duck", "ear", "elephant", "empty", "every", "eye", "face", "fall", "farm", "fast", "feet", "find", "fine", "finger", "finish", "fireman", "first", "fish", "flag", "flower", "food", "for", "frenchfries", "frog", "garbage", "gift", "giraffe", "girl", "give", "glasswindow", "go", "goose", "grandma", "grandpa", "grass", "green", "gum", "hair", "happy", "hat", "hate", "have", "haveto", "head", "hear", "helicopter", "hello", "hen", "hesheit", "hide", "high", "home", "horse", "hot", "hungry", "icecream", "if", "into", "jacket", "jeans", "jump", "kiss", "kitty", "lamp", "later", "like", "lion", "lips", "listen", "look", "loud", "mad", "make", "man", "many", "milk", "minemy", "mitten", "mom", "moon", "morning", "mouse", "mouth", "nap", "napkin", "night", "no", "noisy", "nose", "not", "now", "nuts", "old", "on", "open", "orange", "outside", "owie", "owl", "pajamas", "pen", "pencil", "penny", "person", "pig", "pizza", "please", "police", "pool", "potty", "pretend", "pretty", "puppy", "puzzle", "quiet", "radio", "rain", "read", "red", "refrigerator", "ride", "room", "sad", "same", "say", "scissors", "see", "shhh", "shirt", "shoe", "shower", "sick", "sleep", "sleepy", "smile", "snack", "snow", "stairs", "stay", "sticky", "store", "story", "stuck", "sun", "table", "talk", "taste", "thankyou", "that", "there", "think", "thirsty", "tiger", "time", "tomorrow", "tongue", "tooth", "toothbrush", "touch", "toy", "tree", "uncle", "underwear", "up", "vacuum", "wait", "wake", "water", "wet", "weus", "where", "white", "who", "why", "will", "wolf", "yellow", "yes", "yesterday", "yourself", "yucky", "zebra", "zipper"]
def norm(text):
    return re.sub(r"[^a-z]", "", str(text).lower())
ours = {norm(s): s for s in SIGNEASE_SIGNS}

z = zipfile.ZipFile(ZIP_PATH)
names = z.namelist()
videos = {os.path.basename(n): n for n in names if n.lower().endswith((".mp4", ".mov", ".webm", ".avi"))}
tables = []
for n in names:
    if n.lower().endswith(".csv") and not os.path.basename(n).startswith("."):
        t = pd.read_csv(io.BytesIO(z.read(n)))
        cols = {c.lower(): c for c in t.columns}
        vcol = next((cols[c] for c in cols if "video" in c or "file" in c), None)
        gcol = next((cols[c] for c in cols if "gloss" in c), None)
        pcol = next((cols[c] for c in cols if "participant" in c or "signer" in c or "user" in c), None)
        if vcol and gcol:
            tables.append(pd.DataFrame({
                "video": t[vcol].astype(str).map(os.path.basename),
                "gloss": t[gcol].astype(str),
                "participant": t[pcol].astype(str) if pcol else "",
                "split": os.path.splitext(os.path.basename(n))[0],
            }))
            print("index:", n, len(t), "rows")
assert tables, "No index CSV with video and gloss columns found; send the printed file list to Claude."
manifest = pd.concat(tables).drop_duplicates("video")
manifest["member"] = manifest.video.map(videos)
missing = manifest.member.isna().sum()
manifest = manifest.dropna(subset=["member"])
manifest["signease_sign"] = manifest.gloss.map(lambda g: ours.get(norm(g)) or ours.get(norm(re.sub(r"\d+$", "", str(g)))) or "")
manifest["priority"] = (manifest.signease_sign == "").astype(int)
manifest = manifest.sort_values(["priority", "gloss", "video"]).reset_index(drop=True)
manifest["order"] = range(len(manifest))
manifest.to_csv(os.path.join(DRIVE_DIR, "manifest.csv"), index=False)
found = sorted(set(manifest.signease_sign) - {""})
print(f"{len(manifest)} videos, {manifest.gloss.nunique()} signs ({missing} index rows without a video)")
print(f"SignEase signs found: {len(found)} of 250, {(manifest.priority == 0).sum()} videos")
print("not found by name (Claude will check these by hand):", sorted(set(SIGNEASE_SIGNS) - set(found)))

In [ ]:
WORKER = r'''
# Written by the SignEase notebook (step 5). Runs in worker processes.
import os, tempfile, zipfile
import numpy as np

ROWS = np.array([61, 185, 40, 39, 37, 0, 267, 269, 270, 409, 291, 146, 91, 181, 84, 17, 314, 405, 321, 375, 78, 191, 80, 81, 82, 13, 312, 311, 310, 415, 95, 88, 178, 87, 14, 317, 402, 318, 324, 308, 468, 469, 470, 471, 472, 473, 474, 475, 476, 477, 478, 479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 522, 523, 524, 525, 526, 527, 528, 529, 530, 531, 532, 533, 534, 535, 536, 537, 538, 539, 540, 541, 542, 500, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511])
LAYOUT = (("face_landmarks", 468), ("left_hand_landmarks", 21), ("pose_landmarks", 33), ("right_hand_landmarks", 21))
SETTINGS = dict(static_image_mode=False, model_complexity=1, smooth_landmarks=True,
                refine_face_landmarks=False, min_detection_confidence=0.5, min_tracking_confidence=0.5)
ZIP, MAX_WIDTH = None, 640


def init(zip_path, max_width):
    global ZIP, MAX_WIDTH
    ZIP, MAX_WIDTH = zipfile.ZipFile(zip_path), max_width


def extract(member):
    import cv2
    import mediapipe as mp
    path = os.path.join(tempfile.gettempdir(), "v%d%s" % (os.getpid(), os.path.splitext(member)[1]))
    with open(path, "wb") as f:
        f.write(ZIP.read(member))
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    fps = fps if np.isfinite(fps) and 5 <= fps <= 120 else 30.0
    frames, aspect = [], 0.0
    # A fresh tracker per video, as each recording starts one in the browser.
    with mp.solutions.holistic.Holistic(**SETTINGS) as holistic:
        while True:
            ok, image = cap.read()
            if not ok:
                break
            h, w = image.shape[:2]
            aspect = w / h
            if w > MAX_WIDTH:
                image = cv2.resize(image, (MAX_WIDTH, int(h * MAX_WIDTH / w)), interpolation=cv2.INTER_AREA)
            results = holistic.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
            frame = np.full((543, 2), np.nan, np.float32)
            offset = 0
            for attr, count in LAYOUT:
                lm = getattr(results, attr)
                if lm:
                    frame[offset:offset + count] = [(p.x, p.y) for p in lm.landmark]
                offset += count
            frames.append(frame[ROWS])
    cap.release()
    os.remove(path)
    xy = np.stack(frames).astype(np.float16) if frames else np.zeros((0, len(ROWS), 2), np.float16)
    return member, xy, fps, aspect
'''

# Step 5: extract landmarks. Keep this tab open; it prints progress and ETA.
# Each saved chunk holds, per video: x,y of the 94 landmark rows SignEase
# uses (float16), frame rate and frame aspect ratio.
import importlib, multiprocessing, os, shutil, sys, time
import numpy as np
WORK_DIR = os.path.dirname(os.path.abspath(ZIP_PATH))
with open(os.path.join(WORK_DIR, "signease_extract.py"), "w") as f:
    f.write(WORKER)
sys.path.insert(0, WORK_DIR)
import signease_extract
importlib.reload(signease_extract)

todo = manifest if PHASE == "all" else manifest[manifest.priority == 0]
chunks = [todo.iloc[i:i + CHUNK] for i in range(0, len(todo), CHUNK)]
def chunk_file(chunk):
    return os.path.join(DRIVE_DIR, "chunk_%06d.npz" % int(chunk.order.iloc[0]))
start, done_videos = time.time(), 0
pool = multiprocessing.get_context("spawn").Pool(
    os.cpu_count(), initializer=signease_extract.init, initargs=(ZIP_PATH, MAX_WIDTH))
try:
    for chunk in chunks:
        out = chunk_file(chunk)
        if os.path.exists(out):
            continue
        results = list(pool.imap(signease_extract.extract, chunk.member.tolist(), chunksize=2))
        lengths = np.array([len(r[1]) for r in results])
        local = os.path.join(WORK_DIR, os.path.basename(out))
        np.savez_compressed(
            local, video=np.array([os.path.basename(r[0]) for r in results]), order=chunk.order.to_numpy(),
            lengths=lengths, xy=np.concatenate([r[1] for r in results]),
            fps=np.array([r[2] for r in results], np.float32),
            aspect=np.array([r[3] for r in results], np.float32), rows=signease_extract.ROWS)
        shutil.copy(local, out + ".tmp")
        os.replace(out + ".tmp", out)
        os.remove(local)
        done_videos += len(chunk)
        rate = done_videos / (time.time() - start)
        left = sum(len(c) for c in chunks if not os.path.exists(chunk_file(c)))
        print("saved %s | %.0f videos/hour | %d left, about %.1f h" % (os.path.basename(out), rate * 3600, left, left / rate / 3600))
finally:
    pool.terminate()
print("Phase", PHASE, "finished.")

In [ ]:
# Step 6: status. When the phase is finished, download the whole
# "SignEase/asl_citizen" folder from drive.google.com (right-click → Download)
# and unzip it into SignEase/Ml/training/asl_citizen/ on your Mac.
import glob, numpy as np
files = sorted(glob.glob(os.path.join(DRIVE_DIR, "chunk_*.npz")))
count = sum(len(np.load(f)["video"]) for f in files)
size = sum(os.path.getsize(f) for f in files) / 1e9
print(f"{len(files)} chunks, {count} videos, {size:.2f} GB in Drive")
print(f"SignEase signs: {(manifest.priority == 0).sum()} videos | all: {len(manifest)} videos")